## Opay Bank Statement Parser

In [1]:
import pymupdf
import pandas as pd
import re
import numpy as np

### Custom Wrangling Functions

In [2]:
DATE_PATTERN = re.compile(r"\d{2} [A-Za-z]{3} \d{4}$")
COMPLETE_DATA_MAPPER = re.compile(r'^\d{4}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{2}\s\d{2}:\d{2}:\d{2}$')
FILEPATH = r"opay_bankstatement.pdf"

In [ ]:
def pdf_extractor() -> pd.DataFrame:

    tables = []
    with pymupdf.open(FILEPATH) as doc:
        for page_number in range(len(doc)):
            page = doc.load_page(page_number)
            text_blocks = page.get_text('blocks')

            sorted_blocks = sorted(text_blocks, key=lambda b: b[1])

            table_data = []
            for block in sorted_blocks:
                lines = block[4].split('\n')
                table_data.append(lines)

            if table_data:
                df = pd.DataFrame(table_data)
                processed_df = df

                if not processed_df.empty:
                    tables.append(processed_df)

        if tables:
            concatenated_df = pd.concat(tables, ignore_index=True)
            bank_statement = concatenated_df
        else:
            bank_statement = pd.DataFrame()
        return bank_statement


def check_datetime(row):
    cell_1 = str(row[0])
    return bool(
        re.match(DATE_PATTERN, cell_1)
        or
        re.match(COMPLETE_DATA_MAPPER, cell_1))
        # or 
        # re.match(dt_incomplete_mapper, cell_1))


def clean_df(data: pd.DataFrame) -> pd.DataFrame:
    # Renaming Columns
    col = ['Trans.Time', 'Value Date', 'Description',
           'Debit/Credit(#)', 'Balance(#)',
           'Channel', 'Transaction Reference', 'NoneDrop']
    data.columns = col

    # Dropping multiple headers and rows with irrelevant values.
    bs_df = data.copy()
    bs_df = bs_df.drop(bs_df[bs_df['Trans.Time'] == 'Trans. Time'].index)
    bs_df = bs_df[~bs_df['Trans.Time'].str.contains('^[A-Z]', regex=True)]

    # Ensure Datetime consistency
    bs_df = bs_df[bs_df.apply(check_datetime, axis=1)]

    return bs_df


def shift_and_rejoin_df(data: pd.DataFrame) -> pd.DataFrame:
    # Split dataset into two. 
    nas_dropped = data.dropna(subset=['Balance(#)']).reset_index()
    df_to_shift = nas_dropped[~(nas_dropped['Channel'] == 'E-Channel')].reset_index()
    df_stable = nas_dropped[nas_dropped['Channel'] == 'E-Channel'].reset_index()

    # Selecting Transformation Criteria
    shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()

    for index in shift_criteria.index:
        index = int(index)
        df_to_shift.iloc[index, :] = df_to_shift.iloc[index, :].shift()
    df_shifted = df_to_shift

    final_df = pd.concat([df_stable, df_shifted], axis=0)
    final_df = final_df.drop(
        columns=['level_0', 'index', 'Trans.Time', 'NoneDrop']
        ).reset_index()
    final_df = final_df.drop(columns='index')
    return final_df


def clean_balance_col(data: pd.DataFrame) -> pd.DataFrame:
    data['Debit/Credit(#)'] = pd.to_numeric(
        data['Debit/Credit(#)'].str.replace(
            ',', '').str.replace(
                '+', ''
                ), errors='coerce')
    data['Balance(#)'] = pd.to_numeric(
        data['Balance(#)'].str.replace(
            ',', ''
            ), errors='coerce')

    calculated_balance = data['Balance(#)'].copy()
    # Iterate row-by-row and calculate missing balances
    for i in range(1, len(calculated_balance)):
        if pd.isna(calculated_balance[i]):
            if not pd.isna(calculated_balance[i - 1]) and not pd.isna(data.loc[i, 'Debit/Credit(#)']):
                calculated_balance[i] = round(calculated_balance[i - 1],2) + round(data.loc[i, 'Debit/Credit(#)'],2)
    data['Balance(#)'] = calculated_balance

    data['Credit'] = round(data['Debit/Credit(#)'].apply(lambda x:x if x > 0 else 0), 2)
    data['Debit'] = abs(round(data['Debit/Credit(#)'].apply(lambda x:x if x < 0 else 0), 2))
    data = data.drop(columns='Debit/Credit(#)')
    
    return data


def categorize_transactions(data: pd.DataFrame, bank: str) -> pd.DataFrame:

    data['Description'] = data['Description'].apply(
        lambda x: x.replace('Transfer from ', '').replace('Transfer to ', '')
    )

    conditions = [
        ((data['Description'].isin(['OWealth Deposit(AutoSave)', 'OWealth Deposit'])) & (data['Credit'] > 0)),
        ((data['Description'].isin(['OWealth Deposit(AutoSave)', 'OWealth Deposit'])) & (data['Debit'] > 0)),
        ((data['Description'] == 'OWealth Withdrawal') & (data['Credit'] > 0))
    ]

    choices = [
        'OWealth Deposit',
        'OWealth Withdrawal',
        'OWealth Deposit'
    ]

    data['Description'] = np.select(conditions, choices, default=data['Description'])
    data['Bank'] = bank
    data['Trans_count'] = range(1, len(data['Description'])+1)
    return data



<>:66: SyntaxWarning: invalid escape sequence '\d'
<>:66: SyntaxWarning: invalid escape sequence '\d'
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_2376\3725629806.py:66: SyntaxWarning: invalid escape sequence '\d'
  shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()


### Wrangling Check

In [4]:
ab = pdf_extractor()

ac = clean_df(ab)

ad = shift_and_rejoin_df(ac)

ae = clean_balance_col(ad)

af = categorize_transactions(ae, bank='Opay')

af.head(2)

C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_2376\3725629806.py:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cell_1 = str(row[0])
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_2376\3725629806.py:66: SyntaxWarning: invalid escape sequence '\d'
  shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()


KeyError: 'type'

In [6]:
ad

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference,Credit,Debit
0,12 Mar 2024,Spend & Save Withdrawal,2394.69,2394.69,E-Channel,240312014770779834,2394.69,0.00
1,12 Mar 2024,OWealth Deposit(AutoSave),-294.69,0.00,E-Channel,240312145795962609,0.00,294.69
2,27 Mar 2024,Transfer from NWAGBO CHINEDUM OBIOMA,4500.00,4500.00,E-Channel,000004240327184135633860074199,4500.00,0.00
3,12 Apr 2024,Transfer from EBENEZER AYOMIKUN FALODUN,10000.00,10000.00,E-Channel,000014240412184328237230179735,10000.00,0.00
4,12 Apr 2024,OWealth Deposit(AutoSave),-10000.00,0.00,E-Channel,240412146767712953,0.00,10000.00
...,...,...,...,...,...,...,...,...
739,30 Mar 2025,OWealth Interest Earned,0.09,0.71,E-Channel,2503309971yubCWGb81ei38grs5BvL,0.09,0.00
740,31 Mar 2025,OWealth Interest Earned,0.09,0.80,E-Channel,250331991NjIlD3hc3FB1cfDlCnNp1,0.09,0.00
741,01 Apr 2025,OWealth Interest Earned,0.09,0.89,E-Channel,250401993FeW6pQKUwq8gwbX6pXZ39,0.09,0.00
742,02 Apr 2025,OWealth Interest Earned,0.09,0.98,E-Channel,250402995dD0FEHSCceB5u4PBUOpUQ,0.09,0.00


### Transform cleaned data to specified format.

In [21]:
def transform_df(data: pd.DataFrame)-> pd.DataFrame:
    def transform_row(row):
        return {
            "type": "debit" if int(row["Debit"]) > 0 else "credit",
            "amount": row["Debit"] if int(row["Debit"]) > 0 else row["Credit"],
            "narration": row["Description"].replace(r"'", ""),
            "date": row["Value Date"],
            "balance": row["Balance(#)"],
            "bank": row["Bank"]
        }
    transformed_bs = ae.apply(transform_row, axis=1).tolist()
    transformed_bs_df = pd.DataFrame(transformed_bs)
    return transformed_bs_df

transformed_opay_df = transform_df(ae)
transformed_opay_df.head(2)


,type,amount,narration,date,balance,bank
0,credit,2394.69,Spend & Save Withdrawal,12 Mar 2024,2394.69,Opay
1,debit,294.69,OWealth Withdrawal,12 Mar 2024,0.00,Opay


### Save in file formats (json, excel, parquet, csv)

In [18]:
transformed_opay_df.to_json('data/transformed_opay.json', orient='records')
transformed_opay_df.to_csv('data/transformed_bs.csv')
transformed_opay_df.to_excel('data/transformed_bs.xlsx')
transformed_opay_df.to_parquet('data/transformed_bs.parquet')

In [6]:
df = pd.read_csv(r'C:\Users\APIN PC\OneDrive\Documents\DS\circle_funds_externship\bank_statement_parser\opay_bs\data\transformed_bs.csv')
df['trans_count'] = range(1, len(df['type'])+1)
df

,Unnamed: 0,type,amount,narration,date,balance,bank,trans_count
0,0,credit,2394.69,Spend & Save Withdrawal,12 Mar 2024,2394.69,Opay,1
1,1,debit,294.69,OWealth Withdrawal,12 Mar 2024,0.00,Opay,2
2,2,credit,4500.00,NWAGBO CHINEDUM OBIOMA,27 Mar 2024,4500.00,Opay,3
3,3,credit,10000.00,EBENEZER AYOMIKUN FALODUN,12 Apr 2024,10000.00,Opay,4
4,4,debit,10000.00,OWealth Withdrawal,12 Apr 2024,0.00,Opay,5
...,...,...,...,...,...,...,...,...
739,739,credit,0.09,OWealth Interest Earned,30 Mar 2025,0.71,Opay,740
740,740,credit,0.09,OWealth Interest Earned,31 Mar 2025,0.80,Opay,741
741,741,credit,0.09,OWealth Interest Earned,01 Apr 2025,0.89,Opay,742
742,742,credit,0.09,OWealth Interest Earned,02 Apr 2025,0.98,Opay,743


### Reviewing Profit/Loss Statement

In [9]:
transformed_opay_df.groupby('type')['amount'].sum()

type
credit    1276721.75
debit     1264022.21
Name: amount, dtype: float64